In [ ]:
# ── Standard library ──────────────────────────────────────────────
import os
import sys
import random
import json
from pathlib import Path

# ── Project root ───────────────────────────────────────────────────
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# ── Numerical / data ──────────────────────────────────────────────
import numpy as np

# ── PyTorch core ──────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler, Dataset

# ── Torchvision ───────────────────────────────────────────────────
from torchvision import datasets, models
import torchvision.transforms as T

# ── Augmentation (Albumentations) ─────────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── Progress bar ──────────────────────────────────────────────────
from tqdm import tqdm

# ── Metrics ───────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, average_precision_score,
    RocCurveDisplay, precision_recall_curve
)

# ── Visualization ─────────────────────────────────────────────────
import seaborn as sns
import matplotlib.pyplot as plt

# ── PIL ───────────────────────────────────────────────────────────
from PIL import Image

print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Single source of truth for ALL hyperparameters ────────────────
CFG = {
    # Data
    "data_root":    str(Path(os.environ.get("RGB_DATA_ROOT", REPO_ROOT / "data" / "processed"))),
    "save_dir":     str(Path(os.environ.get("RGB_SAVE_DIR", REPO_ROOT / "results" / "checkpoints"))),
    "save_path":    str(Path(os.environ.get("RGB_SAVE_PATH", REPO_ROOT / "results" / "checkpoints" / "rgb_efficientnet_b4_best.pt"))),

    # Image
    "img_size":     224,

    # Training
    "batch_size":   32,
    "epochs":       15,
    "lr":           1e-4,
    "weight_decay": 1e-4,
    "dropout":      0.3,
    "grad_clip":    1.0,

    # Data split
    "val_split":    0.2,
    "seed":         42,
    "num_workers":  4,         # Kaggle T4 supports 4

    # Model
    "pretrained":   True,
    "num_classes":  2,         # real / fake

    # Imbalance strategy: "sampler" | "loss_weight" | "both"
    # Recommendation: use "sampler" only
    "imbalance_strategy": "sampler",
}

# ── Reproducibility ───────────────────────────────────────────────
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # disable for full repro

set_seed(CFG["seed"])

# ── Device ────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Create output directory ───────────────────────────────────────
os.makedirs(CFG["save_dir"], exist_ok=True)
print(f"Data root : {CFG['data_root']}")
print(f"Save path : {CFG['save_path']}")

In [ ]:
# ── Albumentations adapter for torchvision ImageFolder ────────────
# ImageFolder passes PIL Images; Albumentations expects NumPy uint8.
class AlbumentationsWrapper:
    def __init__(self, transform: A.Compose):
        self.transform = transform

    def __call__(self, img):
        # PIL → NumPy
        img_np = np.array(img)
        result = self.transform(image=img_np)
        return result["image"]   # returns torch.Tensor via ToTensorV2

# ── Training transforms (deepfake-aware augmentations) ────────────
_train_aug = A.Compose([
    A.Resize(CFG["img_size"], CFG["img_size"]),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.2),

    # Color / lighting variations
    A.ColorJitter(
        brightness=0.3, contrast=0.3,
        saturation=0.2, hue=0.1, p=0.7
    ),
    A.ToGray(p=0.05),              # occasional grayscale robustness

    # Compression artifacts — critical for deepfake detection
    # Deepfakes are often re-encoded; model must be robust to this
    A.ImageCompression(quality_lower=50, quality_upper=95, p=0.5),

    # Sensor noise simulation
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),

    # Occlusion robustness
    A.CoarseDropout(
        max_holes=6, max_height=24, max_width=24,
        min_holes=1, fill_value=0, p=0.3
    ),

    # Geometric
    A.ShiftScaleRotate(
        shift_limit=0.05, scale_limit=0.1,
        rotate_limit=10, p=0.4
    ),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

# ── Validation transforms (no augmentation, only normalize) ───────
_val_aug = A.Compose([
    A.Resize(CFG["img_size"], CFG["img_size"]),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

train_tfms = AlbumentationsWrapper(_train_aug)
val_tfms   = AlbumentationsWrapper(_val_aug)

print("Transforms ready.")

In [ ]:
# ── Load once without transform to get indices and targets ─────────
full_ds = datasets.ImageFolder(CFG["data_root"])

print(f"Total images  : {len(full_ds)}")
print(f"Classes       : {full_ds.classes}")
print(f"Class → index : {full_ds.class_to_idx}")

# Verify expected structure
assert set(full_ds.classes) == {"real", "fake"} or len(full_ds.classes) == 2, \
    f"Expected 2 classes (real/fake), got: {full_ds.classes}"

# ── Reproducible random split ─────────────────────────────────────
generator = torch.Generator().manual_seed(CFG["seed"])
indices   = torch.randperm(len(full_ds), generator=generator).tolist()

split     = int(len(indices) * (1 - CFG["val_split"]))
train_idx = indices[:split]
val_idx   = indices[split:]

print(f"\nTrain samples : {len(train_idx)}")
print(f"Val samples   : {len(val_idx)}")

# ── Apply transforms via two ImageFolder instances ─────────────────
# Critical: two separate instances so train and val get
# different transforms while sharing the same indices.
train_ds = Subset(
    datasets.ImageFolder(CFG["data_root"], transform=train_tfms),
    train_idx
)
val_ds = Subset(
    datasets.ImageFolder(CFG["data_root"], transform=val_tfms),
    val_idx
)

# ── Class imbalance stats ─────────────────────────────────────────
train_targets = np.array(full_ds.targets)[train_idx]
class_counts  = np.bincount(train_targets, minlength=CFG["num_classes"])
class_weights_np = 1.0 / np.maximum(class_counts, 1)

print(f"\nClass counts (train): {dict(zip(full_ds.classes, class_counts))}")
print(f"Class weights       : {dict(zip(full_ds.classes, class_weights_np.round(6)))}")

In [ ]:
# ── WeightedRandomSampler (train only) ────────────────────────────
sample_weights = class_weights_np[train_targets]

sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

# ── DataLoaders ───────────────────────────────────────────────────
train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    sampler=sampler,              # mutually exclusive with shuffle=True
    num_workers=CFG["num_workers"],
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=True,      # keeps worker processes alive between epochs
    prefetch_factor=2,
)

val_loader = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=True,
)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

# ── Sanity check: visualize one batch ─────────────────────────────
images, labels = next(iter(train_loader))
print(f"\nBatch shape   : {images.shape}")   # (32, 3, 224, 224)
print(f"Label sample  : {labels[:8].tolist()}")
print(f"Pixel range   : [{images.min():.2f}, {images.max():.2f}]")

In [ ]:
class RGBDeepfakeClassifier(nn.Module):
    """
    EfficientNet-B4 backbone with a custom two-layer classification head.
    Pretrained on ImageNet; head fine-tuned for binary deepfake detection.
    """
    def __init__(self, num_classes: int = 2, dropout: float = 0.3, pretrained: bool = True):
        super().__init__()
        weights = models.EfficientNet_B4_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.efficientnet_b4(weights=weights)

        # ── Store the in_features BEFORE replacing the classifier ──────
        in_features = backbone.classifier[1].in_features   # 1792 for EfficientNet-B4

        # ── Remove the original classification head ─────────────────
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        # Output shape: (B, 1792, 1, 1)

        # ── Custom head: Dropout → FC → ReLU → Dropout → FC ────────
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout * 0.7),   # lighter second dropout
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)   # (B, 1792, 1, 1)
        return self.head(features)    # (B, num_classes)

    def freeze_backbone(self):
        """Freeze backbone for the first N epochs (optional warm-up)."""
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True


# ── Instantiate ───────────────────────────────────────────────────
model = RGBDeepfakeClassifier(
    num_classes=CFG["num_classes"],
    dropout=CFG["dropout"],
    pretrained=CFG["pretrained"],
).to(DEVICE)

# ── Parameter count ───────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

In [ ]:
# ── Loss ──────────────────────────────────────────────────────────
# Using sampler-only strategy (CFG["imbalance_strategy"] == "sampler")
# If you want weighted loss instead, set class_weights_tensor here.
if CFG["imbalance_strategy"] == "loss_weight":
    cw = torch.tensor(class_weights_np, dtype=torch.float32).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=cw)
else:
    # Sampler already handles imbalance — plain CE avoids double correction
    criterion = nn.CrossEntropyLoss()

# ── Optimizer ─────────────────────────────────────────────────────
# Differential LR: lower LR for pretrained backbone, higher for new head
optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(), "lr": CFG["lr"] * 0.1},
    {"params": model.head.parameters(),     "lr": CFG["lr"]},
], weight_decay=CFG["weight_decay"])

# ── Scheduler: CosineAnnealingLR ──────────────────────────────────
# Steps every BATCH (not epoch) for smooth decay
total_steps = CFG["epochs"] * len(train_loader)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=total_steps,
    eta_min=1e-6,
)

# ── Mixed precision scaler ────────────────────────────────────────
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

print(f"Criterion : {criterion}")
print(f"Optimizer : AdamW (backbone lr={CFG['lr']*0.1:.1e}, head lr={CFG['lr']:.1e})")
print(f"Scheduler : CosineAnnealingLR over {total_steps} steps")

In [ ]:
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    threshold: float = 0.5,
) -> dict:
    """
    Full evaluation: loss, accuracy, AUC-ROC, AUC-PR, F1.
    Returns a dict so callers can log any subset of metrics.
    """
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs:  list[float] = []
    all_labels: list[int]   = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                logits = model(images)
                loss   = criterion(logits, labels)

            probs = torch.softmax(logits, dim=1)[:, 1]  # P(fake)
            total_loss += loss.item() * labels.size(0)
            correct    += (logits.argmax(dim=1) == labels).sum().item()
            total      += labels.size(0)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    preds = [1 if p >= threshold else 0 for p in all_probs]

    return {
        "loss":    total_loss / total,
        "acc":     correct / total,
        "auc_roc": roc_auc_score(all_labels, all_probs),
        "auc_pr":  average_precision_score(all_labels, all_probs),
        "f1":      f1_score(all_labels, preds, zero_division=0),
        "probs":   np.array(all_probs),
        "labels":  np.array(all_labels),
    }

print("evaluate() function defined.")

In [ ]:
Path(CFG["save_dir"]).mkdir(parents=True, exist_ok=True)

In [ ]:
history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_auc": [], "val_f1": []}
best_val_auc = 0.0
early_stop_patience = 5
epochs_no_improve   = 0

for epoch in range(1, CFG["epochs"] + 1):

    # ── Optional: freeze backbone for epoch 1, unfreeze after ──────
    if epoch == 1:
        model.freeze_backbone()
        print("Epoch 1: backbone frozen (head warm-up)")
    elif epoch == 2:
        model.unfreeze_backbone()
        print("Epoch 2+: full model unfrozen")

    # ── Train ─────────────────────────────────────────────────────
    model.train()
    running_loss, seen = 0.0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch:02d}/{CFG['epochs']}"):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(images)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()   # step per batch for cosine

        running_loss += loss.item() * labels.size(0)
        seen         += labels.size(0)

    train_loss = running_loss / seen

    # ── Validate ──────────────────────────────────────────────────
    val_metrics = evaluate(model, val_loader, criterion, DEVICE)

    # ── Log ───────────────────────────────────────────────────────
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_metrics["loss"])
    history["val_acc"].append(val_metrics["acc"])
    history["val_auc"].append(val_metrics["auc_roc"])
    history["val_f1"].append(val_metrics["f1"])

    current_lr = optimizer.param_groups[-1]["lr"]   # head LR
    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_acc={val_metrics['acc']:.4f} | "
        f"val_auc={val_metrics['auc_roc']:.4f} | "
        f"val_f1={val_metrics['f1']:.4f} | "
        f"lr={current_lr:.2e}"
    )

    # ── Checkpoint on best AUC ────────────────────────────────────
    if val_metrics["auc_roc"] > best_val_auc:
        best_val_auc    = val_metrics["auc_roc"]
        epochs_no_improve = 0
        torch.save({
            "epoch":              epoch,
            "model_state_dict":   model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_acc":            val_metrics["acc"],
            "val_auc":            val_metrics["auc_roc"],
            "val_f1":             val_metrics["f1"],
            "class_to_idx":       full_ds.class_to_idx,
            "cfg":                CFG,
        }, CFG["save_path"])
        print(f"  ✓ Saved checkpoint (AUC={best_val_auc:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= early_stop_patience:
            print(f"Early stopping triggered at epoch {epoch}.")
            break

print(f"\nTraining complete. Best val AUC: {best_val_auc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs_ran = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs_ran, history["train_loss"], label="Train Loss")
axes[0].plot(epochs_ran, history["val_loss"],   label="Val Loss")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")

axes[1].plot(epochs_ran, history["val_auc"],    label="Val AUC-ROC", color="green")
axes[1].plot(epochs_ran, history["val_f1"],     label="Val F1",      color="orange")
axes[1].set_title("AUC & F1"); axes[1].legend(); axes[1].set_xlabel("Epoch")

axes[2].plot(epochs_ran, history["val_acc"],    label="Val Accuracy", color="purple")
axes[2].set_title("Accuracy"); axes[2].legend(); axes[2].set_xlabel("Epoch")

plt.tight_layout()
plt.savefig("/kaggle/working/training_curves.png", dpi=150)
plt.show()
print("Curves saved.")

In [ ]:
# ── Load best checkpoint ──────────────────────────────────────────
checkpoint = torch.load(CFG["save_path"], map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded epoch {checkpoint['epoch']} | AUC={checkpoint['val_auc']:.4f}")

# ── Re-evaluate on val set ────────────────────────────────────────
val_metrics = evaluate(model, val_loader, criterion, DEVICE)
probs  = val_metrics["probs"]
labels = val_metrics["labels"]
preds  = (probs >= 0.5).astype(int)

class_names = list(full_ds.class_to_idx.keys())

# ── Confusion matrix ──────────────────────────────────────────────
cm = confusion_matrix(labels, preds)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix")

# ── ROC Curve ─────────────────────────────────────────────────────
RocCurveDisplay.from_predictions(labels, probs, ax=axes[1], name="EfficientNet-B4 RGB")
axes[1].set_title(f"ROC Curve (AUC={val_metrics['auc_roc']:.4f})")
axes[1].plot([0, 1], [0, 1], "k--")

# ── Precision-Recall Curve ────────────────────────────────────────
prec, rec, _ = precision_recall_curve(labels, probs)
axes[2].plot(rec, prec, color="darkorange")
axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision")
axes[2].set_title(f"PR Curve (AUC-PR={val_metrics['auc_pr']:.4f})")

plt.tight_layout()
plt.savefig(Path(CFG["save_dir"]) / "evaluation_plots.png", dpi=150)
plt.show()

# ── Summary table ─────────────────────────────────────────────────
print("\n── Final Metrics ──────────────────────────────")
print(f"  Accuracy : {val_metrics['acc']:.4f} ({val_metrics['acc']*100:.1f}%)")
print(f"  AUC-ROC  : {val_metrics['auc_roc']:.4f}")
print(f"  AUC-PR   : {val_metrics['auc_pr']:.4f}")
print(f"  F1 Score : {val_metrics['f1']:.4f}")

# Per-class breakdown
tn, fp, fn, tp = cm.ravel()
print(f"\n  True Positives  (Fake→Fake)  : {tp}")
print(f"  True Negatives  (Real→Real)  : {tn}")
print(f"  False Positives (Real→Fake)  : {fp}")
print(f"  False Negatives (Fake→Real)  : {fn}")
print(f"  Precision (Fake): {tp/(tp+fp):.4f}")
print(f"  Recall    (Fake): {tp/(tp+fn):.4f}")

In [ ]:
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image


def generate_gradcam(
    model,
    image_path,
    device,
    transform,
    target_layer_name=None,
    figsize=(12, 4)
) -> None:

    activations = []
    gradients = []

    def forward_hook(module, inp, out):
        activations.clear()
        activations.append(out.detach())

    def backward_hook(module, grad_in, grad_out):
        gradients.clear()
        gradients.append(grad_out[0].detach())

    if target_layer_name is None:
        conv_layers = [
            (name, module)
            for name, module in model.named_modules()
            if isinstance(module, torch.nn.Conv2d)
        ]
        if not conv_layers:
            raise ValueError("No convolution layers found for Grad-CAM.")
        target_layer_name, target_layer = conv_layers[-1]
    else:
        target_layer = dict(model.named_modules())[target_layer_name]

    fwd_handle = target_layer.register_forward_hook(forward_hook)
    bwd_handle = target_layer.register_full_backward_hook(backward_hook)

    img = Image.open(image_path).convert("RGB")
    orig_np = np.array(img)

    if hasattr(transform, "transform"):
        aug = transform.transform(image=orig_np)
        tensor = aug["image"].unsqueeze(0).to(device)
    else:
        tensor = transform(img).unsqueeze(0).to(device)

    tensor.requires_grad_(True)
    model.eval()

    with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
        logits = model(tensor)

    probs = torch.softmax(logits, dim=1).squeeze()
    pred_idx = int(logits.argmax(dim=1).item())
    pred_score = probs[pred_idx].item()

    model.zero_grad()
    logits[0, pred_idx].backward()

    grads = gradients[0].squeeze(0)   # (C,H,W)
    acts  = activations[0].squeeze(0)

    weights = grads.mean(dim=(1, 2))
    cam = (weights[:, None, None] * acts).sum(dim=0)
    cam = torch.relu(cam)

    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)
    cam_np = np.nan_to_num(cam.cpu().numpy()).astype(np.float32)

    if cam_np.size == 0:
        raise ValueError("Grad-CAM is empty")

    H, W = orig_np.shape[:2]
    cam_resized = cv2.resize(cam_np, (W, H))
    heatmap = (plt.cm.jet(cam_resized)[:, :, :3] * 255).astype(np.uint8)
    overlay = (0.45 * heatmap + 0.55 * orig_np).astype(np.uint8)

    fwd_handle.remove()
    bwd_handle.remove()

    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.05)

    for i, (img_show, title) in enumerate([
        (orig_np, "Original"),
        (heatmap, "Grad-CAM"),
        (overlay, "Overlay"),
    ]):
        ax = fig.add_subplot(gs[i])
        ax.imshow(img_show)
        ax.set_title(title)
        ax.axis("off")

    plt.suptitle(
        f"Prediction: {pred_idx} | Confidence: {pred_score:.3f} | Layer: {target_layer_name}",
        fontsize=13
    )

    plt.show()
    print("Grad-CAM generated successfully")

In [ ]:
def predict_image(
    image_path: str,
    model: nn.Module,
    device: torch.device,
    img_size: int = 224,
    threshold: float = 0.5,
) -> dict:
    """
    Run inference on one image file.
    Returns predicted class, confidence, and raw probabilities.
    """
    # ── Load & preprocess ─────────────────────────────────────────
    img = Image.open(image_path).convert("RGB")
    img_np = np.array(img)

    transform = A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    tensor = transform(image=img_np)["image"].unsqueeze(0).to(device)

    # ── Inference ─────────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits = model(tensor)
    probs = torch.softmax(logits, dim=1).squeeze().cpu().numpy()

    # ── Map to class names ────────────────────────────────────────
    idx_to_class = {v: k for k, v in full_ds.class_to_idx.items()}
    fake_idx     = full_ds.class_to_idx.get("fake", 1)
    pred_class   = idx_to_class[int(probs.argmax())]
    confidence   = float(probs.max())
    fake_prob    = float(probs[fake_idx])

    # ── Visualize ─────────────────────────────────────────────────
    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    color = "red" if fake_prob >= threshold else "green"
    plt.title(
        f"Predicted: {pred_class.upper()} | "
        f"P(fake)={fake_prob:.3f} | "
        f"Confidence={confidence:.3f}",
        color=color, fontsize=11
    )
    plt.axis("off")
    plt.show()

    return {
        "prediction": pred_class,
        "confidence": confidence,
        "p_fake":     fake_prob,
        "p_real":     float(probs[1 - fake_idx]),
    }


sample_image = os.environ.get("RGB_SAMPLE_IMAGE")
if sample_image and Path(sample_image).exists():
    result = predict_image(sample_image, model, DEVICE)
    print(result)
else:
    print("Set RGB_SAMPLE_IMAGE to run the inference demo.")

print("predict_image() ready.")

In [ ]:
checkpoint = torch.load(CFG["save_path"], weights_only=False, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])

sample_image = os.environ.get("RGB_SAMPLE_IMAGE")
if sample_image and Path(sample_image).exists():
    generate_gradcam(
        model=model,
        image_path=sample_image,
        device=DEVICE,
        transform=val_tfms,
        target_layer_name=None,
    )
else:
    print("Set RGB_SAMPLE_IMAGE to run the Grad-CAM demo.")